In [0]:
# Consolidação: landing (Parquet) -> bronze (Delta)
#   /Volumes/meu_catalog/landing/postgres/unifor/{tabela}/YYYY/YYYY-MM/YYYY-MM-DD/*.parquet
import re
from datetime import datetime, date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SOURCE_DATABASE = "unifor"
BASE_VOLUME     = "/Volumes/meu_catalog/landing/postgres"
BRONZE_SCHEMA   = "meu_catalog.bronze"
TABLE_PREFIX    = "unifor__"

# intervalo de datass
START_DATE = datetime.strptime("01/08/2026", "%d/%m/%Y").date()
END_DATE   = datetime.strptime("02/08/2026", "%d/%m/%Y").date()

AUDIT_ORDER_COL = "_ingested_at" 
_INVALID_CHARS = re.compile(r"[ ,;{}()\n\t=]+") 

PRIMARY_KEYS = {
    "actor":    ["actor_id"],
    "film":     ["film_id"],
    "customer": ["customer_id"],
    "payment":  ["payment_id"],
    "rental":   ["rental_id"],
}

def date_range(start: date, end: date):
    d = start
    while d <= end:
        yield d
        d += timedelta(days=1)

def list_tables(base_db_dir: str):
    return sorted(f.name.rstrip("/") for f in dbutils.fs.ls(base_db_dir))

def parquet_paths_for_table(tbl: str, start: date, end: date):
    paths = []
    for d in date_range(start, end):
        day_dir = f"{BASE_VOLUME}/{SOURCE_DATABASE}/{tbl}/{d:%Y}/{d:%Y-%m}/{d:%Y-%m-%d}"
        try:
            paths += [
                f.path for f in dbutils.fs.ls(day_dir)
                if f.name.endswith(".parquet") and not f.name.startswith("_")
            ]
        except Exception:
            pass  
    return paths

def deduplicate(df, tbl: str):
    audit_cols    = [c for c in df.columns if c.startswith("_")]
    business_cols = [c for c in df.columns if c not in audit_cols]

    keys = PRIMARY_KEYS.get(tbl, business_cols)

    order = (
        F.col(AUDIT_ORDER_COL).desc()
        if AUDIT_ORDER_COL in df.columns else F.lit(1)
    )
    w = Window.partitionBy(*keys).orderBy(order)

    return (
        df.withColumn("_rn", F.row_number().over(w))
          .where("_rn = 1")
          .drop("_rn")
    )

def sanitize_columns(df):
    seen, renames = {}, []
    for old in df.columns:
        clean = _INVALID_CHARS.sub("_", old).strip("_") or "col"
        base, i = clean, 1
        while clean in seen:
            i += 1
            clean = f"{base}_{i}"
        seen[clean] = True
        renames.append((old, clean))
 
    for old, clean in renames:
        if old != clean:
            df = df.withColumnRenamed(old, clean)
    return df    

In [0]:
tables = list_tables(f"{BASE_VOLUME}/{SOURCE_DATABASE}/")
print(f"Intervalo: {START_DATE:%d/%m/%Y} a {END_DATE:%d/%m/%Y}")
print(f"{len(tables)} tabela(s): {tables}")
 
resultados = []
 
for tbl in tables:
    target_table = f"{BRONZE_SCHEMA}.{TABLE_PREFIX}{tbl}"
    try:
        paths = parquet_paths_for_table(tbl, START_DATE, END_DATE)
        if not paths:
            print(f"[SKIP] {tbl:<20} sem arquivos no intervalo")
            resultados.append((tbl, "SKIP", 0, 0, 0, target_table, ""))
            continue
 
        raw = spark.read.option("mergeSchema", "true").parquet(*paths)
        raw   = sanitize_columns(raw)
        n_raw = raw.count()
 
        dedup = deduplicate(raw, tbl)
        n_dedup = dedup.count()
 
        (
            dedup.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )
 
        removidos = n_raw - n_dedup
        print(f"[OK]   {tbl:<20} arquivos={len(paths):<3} "
              f"lidos={n_raw:<8} dedup={n_dedup:<8} removidos={removidos:<6} -> {target_table}")
        resultados.append((tbl, "OK", len(paths), n_raw, n_dedup, target_table, ""))
 
    except Exception as e:
        print(f"[ERRO] {tbl:<20} -> {str(e)[:200]}")
        resultados.append((tbl, "ERRO", 0, 0, 0, target_table, str(e)[:500]))

In [0]:

resumo = spark.createDataFrame(
    resultados,
    schema="table_name string, status string, arquivos int, "
           "linhas_lidas long, linhas_bronze long, target_table string, error string"
)

resumo.orderBy("status", "table_name").show(truncate=False)

ok   = resumo.where("status = 'OK'").count()
skip = resumo.where("status = 'SKIP'").count()
err  = resumo.where("status = 'ERRO'").count()
print(f"Concluído: {ok} OK / {skip} sem dados / {err} com erro")